```
 pip install tree-sitter tree-sitter-cuda
 ```

In [ ]:
import tree_sitter_cuda, tree_sitter_cpp
from tree_sitter import Language, Parser, Query, QueryCursor
import cj_codetools.parse as parse
import cj_codetools.generator as gen

CUDA = Language(tree_sitter_cuda.language())
CPP = Language(tree_sitter_cpp.language())
parser = Parser(CUDA)

%load_ext autoreload
%autoreload 2

In [ ]:
file = "../custom_jax_cuda/fmm.cuh"

with open(file, 'r') as f:
    txt = f.read()
tree = parser.parse(txt.encode())
root_node = tree.root_node

all_funcs = parse.get_functions(tree.root_node, txt)

all_funcs["TestPositions"].template_par[0].instances = [0,1,2,3]

funcs = [all_funcs["TestPositions"], all_funcs["SimpleArange"]]
includes = ["../fmm.cuh"]

module_code = gen.create_ffi_module(funcs, includes)

outfile = "../custom_jax_cuda/generated/ffi_fmm.cu"
with open(outfile, 'w') as f:
    f.write(module_code)

In [ ]:
# myfunc = tuple(filter(lambda f: f.name == "MySimpleKernel", funcs))[0]
# myfunc = funcs["TestPositions"]
# myfunc.template_par[0].instances = [0,1,2,3]

# print(create_ffi_call(myfunc))